In [15]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import importlib
import joblib
import quantum_function
importlib.reload(quantum_function)

from quantum_function import *
from IPython.display import display, Markdown

In [16]:
# --- SAFETY SWITCH: esce subito se non abiliti esplicitamente ---
RUN_THIS_CELL = True   # metti True solo quando vuoi davvero eseguire

if not RUN_THIS_CELL:
    display(Markdown("Cell locked. Set `RUN_THIS_CELL = True` to run"))
    raise SystemExit



plt.rcParams["text.usetex"] = True
plt.rcParams.update({
    "mathtext.fontset": "cm",      # font simile a LaTeX
    "font.family": "serif",
    "font.size": 14,
    "axes.unicode_minus": False
})

In [ ]:
#PARAMETERS!

gamma = 1        # Atom decay rate
T_1 = 1 / gamma  # time constant for the decay

step = 0.01
endpoint = 20 
times = np.array(np.arange(0, endpoint*T_1, step*T_1)) 

#my_label_eta = ["1", "0.9", "0.7", "0.5", "0.3"] 
my_label_eta = ["1"]

#my_eta = [1, 0.9, 0.7, 0.5, 0.3] #etas for the homodyne detection
my_eta = [1] #etas for the homodyne detection

css_theta = np.pi/4  # angle for the CSS state
state = css_2(css_theta, phi=0)

my_opts_dict = {
    "keep_runs_results": False,
    #"map":"parallel",
    #"normalize_output" : False,
    #"store_final_state" : False,
    #"store_states": False,
    #"store_measurement": False,
    #"progress_bar": "Enhanced",
    "num_cpus": os.cpu_count()-1,
    }

columns_label = [
                #"Energy",
                 "Conc",
                 "Xi2_KU",
                 #"F_phi_plus",
                 #"F_phi_minus",
                 #"F_psi_plus",
                 #"F_psi_minus",
                "Variance_z"
            ]

label_graph_column = {
    #"Energy": r"Energy",
    "Conc": r"$\overline{\mathcal{C}}$",
    "Xi2_KU": r"$\xi^2_{KU}$",
    #"F_phi_plus": r"Fidelity $|\Phi^+\rangle$",
    #"F_phi_minus": r"Fidelity $|\Phi^-\rangle$",
    #"F_psi_plus": r"Fidelity $|\Psi^+\rangle$",
    #"F_psi_minus": r"Fidelity $|\Psi^-\rangle$",
    "Variance_z": r"$\mathrm{Var}(J_z)$"
}

my_e_ops = [
    #energy_solver, 
    concurrence_for_solver_general, 
    xi_KU_solver,
    Variance_z
    ]

In [21]:
from joblib import Parallel, delayed, parallel_backend

N_JOBS = max(1, os.cpu_count() - 1)

def mc_chunk(phi1, phi2, eta, ntraj_chunk, seed):
    # seed diverso per ogni processo (evita traiettorie duplicate)
    np.random.seed(seed)

    opts_local = dict(my_opts_dict)
    opts_local["num_cpus"] = 1
    opts_local.pop("map", None)

    L_unobs = qutip.liouvillian(H_free, collapsing_operators(gamma, phi1, phi2, 1 - eta))
    c_ops_obs = collapsing_operators(gamma, phi1, phi2, eta)

    sol = mcsolve(
        L_unobs,
        state,
        times,
        c_ops=c_ops_obs,
        e_ops=[],
        ntraj=ntraj_chunk,
        options=opts_local
    )

    expect_avg = np.array(sol.expect)  # (n_eops, n_times) già media sul chunk
    return expect_avg, ntraj_chunk


def mc_parallel(phi1, phi2, eta, ntraj_total):
    # spezza in chunk
    chunks = np.array_split(np.arange(ntraj_total), N_JOBS)
    ntrajs = [len(c) for c in chunks if len(c) > 0]

    # seeds indipendenti
    ss = np.random.SeedSequence(12345)
    seeds = ss.spawn(len(ntrajs))
    seeds = [int(s.generate_state(1)[0]) for s in seeds]

    with parallel_backend("loky", inner_max_num_threads=1):
        out = Parallel(n_jobs=len(ntrajs), batch_size=1, verbose=10)(
            delayed(mc_chunk)(phi1, phi2, eta, n_chunk, seed)
            for n_chunk, seed in zip(ntrajs, seeds)
        )

    # combina le medie pesate
    num = None
    den = 0
    for expect_avg, n_chunk in out:
        if num is None:
            num = expect_avg * n_chunk
        else:
            num += expect_avg * n_chunk
        den += n_chunk

    expect_total = num / den
    return pd.DataFrame(expect_total.T, columns=columns_label)


In [22]:
my_phis =[
    [0,0], 
    #[0, np.pi/2], 
    #[np.pi/2,0], 
    #[np.pi/2, np.pi/2]
    ]

ntraj = 50  # o quello che vuoi

for phi1, phi2 in my_phis:
    ineff_df = {}

    for eta, label in zip(my_eta, my_label_eta):

        ineff_df[label] = mc_parallel(phi1, phi2, eta, ntraj)

        my_photodetection = mcsolve(
            qutip.liouvillian(H_free, collapsing_operators(gamma, phi1, phi2, 1-eta)), 
            state, times,

            c_ops = collapsing_operators(gamma, phi1, phi2, eta),
            
            e_ops=my_e_ops,
            ntraj=ntraj,
            options=my_opts_dict
        )

        ineff_df[f"{label}"] = pd.DataFrame(np.transpose(my_photodetection.expect), columns=columns_label)

    
    #Setting the path to save the graphs
    phi1_dir = angle_to_path(phi1)
    phi2_dir = angle_to_path(phi2)
    my_directory = f".\\Graphs\\Jz_3_Ineff_Photodetection_SIM_xi_100\\phi_1={phi1_dir}_phi_2={phi2_dir}"
    Path(my_directory).mkdir(parents=True, exist_ok=True)

    #My theoretical curves
    theo_curves = [np.sin(css_theta)**2 * np.exp(-(1-eta)*times)*(1-np.exp(-gamma*eta/2*times)) for eta in my_eta]

    #Plotting!
    for e_op in columns_label:
        #Un graph per ogni e_op, al variare di eta
        plt.figure(figsize=(12,8))
        graph_name = f"Ineff_{e_op}"

        for (label, df), eta, theory in zip(ineff_df.items(), my_eta, theo_curves):
            # x: tempo normalizzato a T1, y: media delle traiettorie
            x = times/T_1
            y = df[e_op]
            line_sim, = plt.plot(x, y, label=r"$\eta$ = " + str(eta))
            #plt.plot(times/T_1, theory, '--', color=line_sim.get_color(), alpha=0.5, linewidth=2)

        #plt.ylim(-0.01, 1.02)
        plt.xlim(0,endpoint)

        #plt.axhline(1, color="k", linewidth=1.6, linestyle="-", zorder=5)
        #plt.axhspan(1, 1.05, color="k", alpha=0.7, zorder=1, label="_nolegend_")

        plt.xlabel(r"$t/T_1$")
        plt.ylabel(label_graph_column[e_op])

        phi1_tex = angle_to_tex(phi1)
        phi2_tex = angle_to_tex(phi2)

        plt.title(r"Inefficient Photodetection: Average " + label_graph_column[e_op]+ r" over time"+ rf"$, \ \phi_1={phi1_tex}\ \ \phi_2={phi2_tex}$"
            + rf"  (n$_\mathrm{{traj}}$ = {ntraj})")
        plt.grid(True, linestyle=":", alpha=0.6)
        plt.legend(loc="upper left", bbox_to_anchor=(0, 0.95), fontsize=12)

        plt.savefig(Path(my_directory) / f"{graph_name}.pdf", bbox_inches="tight")
        plt.show()

TypeError: 'int' object is not iterable